# Model Pruning with Fisher Information

This notebook demonstrates how to use Fisher information for intelligent model pruning.

## What You'll Learn
- Understanding Fisher information for parameter importance
- Collecting Fisher information during training
- Generating pruning masks at different granularities
- Comparing pruned vs. original models
- Structured vs. unstructured pruning

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from llm_hooks.core import HookManager
from llm_hooks.fisher import FisherHook
from llm_hooks.utils import count_parameters, get_model_size
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style('whitegrid')
%matplotlib inline

## What is Fisher Information?

Fisher information measures how much a parameter affects the model's predictions:
- **High Fisher score**: Parameter is important, shouldn't be pruned
- **Low Fisher score**: Parameter has little effect, safe to prune

Mathematically: $F_\theta = \mathbb{E}[\nabla_\theta \log p(y|x)^2]$

In practice, we approximate it using gradient squared: $F_\theta \approx g_\theta^2$

## Step 1: Create a Model to Prune

In [ ]:
class ConvNet(nn.Module):
    """Simple CNN for demonstration"""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(256 * 4 * 4, 512)
        self.fc2 = nn.Linear(512, 10)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Create model
model = ConvNet()

# Print model stats
num_params = count_parameters(model)
model_size = get_model_size(model, unit='MB')

print(f"Model created:")
print(f"  Parameters: {num_params:,}")
print(f"  Size: {model_size:.2f} MB")

## Step 2: Set Up Fisher Hook

We'll use `FisherHook` to accumulate Fisher information during training.

In [ ]:
# Create hook manager
manager = HookManager(name="fisher_pruning")

# Register Fisher hook
fisher_hook = FisherHook(
    accumulate=True,  # Accumulate Fisher scores over batches
)

manager.register(fisher_hook)
manager.apply_to_model(model)

print("✓ Fisher hook registered and applied")

## Step 3: Collect Fisher Information

Train for a few iterations to accumulate Fisher scores.

In [ ]:
# Create dummy training data (32x32 RGB images)
batch_size = 16
num_batches = 20

train_data = [
    (torch.randn(batch_size, 3, 32, 32), torch.randint(0, 10, (batch_size,)))
    for _ in range(num_batches)
]

# Setup training
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Training loop to collect Fisher information
model.train()
print("Collecting Fisher information...")

for batch_idx, (data, target) in enumerate(train_data):
    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, target)
    loss.backward()  # Fisher hook captures gradients here
    optimizer.step()
    
    if (batch_idx + 1) % 5 == 0:
        print(f"  Batch {batch_idx + 1}/{num_batches}, Loss: {loss.item():.4f}")

print("\n✓ Fisher information collected")

## Step 4: Analyze Fisher Scores

Let's examine the Fisher information for each parameter.

In [ ]:
# Get Fisher scores
fisher_scores = fisher_hook.get_fisher_scores()

print(f"Fisher scores collected for {len(fisher_scores)} parameters:\n")

# Analyze Fisher scores
for param_name, fisher_value in fisher_scores.items():
    if isinstance(fisher_value, torch.Tensor):
        mean_fisher = fisher_value.mean().item()
        max_fisher = fisher_value.max().item()
        print(f"  {param_name:40s} - Mean: {mean_fisher:10.4f}, Max: {max_fisher:10.4f}")

## Step 5: Visualize Parameter Importance

Let's visualize which parameters are most/least important.

In [ ]:
# Extract mean Fisher scores
param_names = []
mean_fisher_scores = []

for param_name, fisher_value in fisher_scores.items():
    if isinstance(fisher_value, torch.Tensor) and 'weight' in param_name:
        param_names.append(param_name.replace('.weight', ''))
        mean_fisher_scores.append(fisher_value.mean().item())

# Plot
plt.figure(figsize=(12, 6))
bars = plt.barh(range(len(param_names)), mean_fisher_scores, color='steelblue')

# Highlight layers with low importance (candidates for aggressive pruning)
threshold = np.median(mean_fisher_scores)
for i, score in enumerate(mean_fisher_scores):
    if score < threshold:
        bars[i].set_color('coral')

plt.yticks(range(len(param_names)), param_names)
plt.xlabel('Mean Fisher Score (Importance)')
plt.title('Parameter Importance Based on Fisher Information\n(Coral = Low importance, candidates for pruning)')
plt.xscale('log')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## Step 6: Generate Pruning Masks

Now let's create pruning masks at different granularities.

In [ ]:
# Test different pruning ratios and granularities
pruning_configs = [
    {"ratio": 0.3, "granularity": "weight", "name": "30% Weight Pruning"},
    {"ratio": 0.5, "granularity": "weight", "name": "50% Weight Pruning"},
    {"ratio": 0.3, "granularity": "neuron", "name": "30% Neuron Pruning"},
]

print("Generating pruning masks...\n")

for config in pruning_configs:
    masks = fisher_hook.get_pruning_mask(
        pruning_ratio=config["ratio"],
        granularity=config["granularity"]
    )
    
    # Calculate actual pruning statistics
    total_params = 0
    pruned_params = 0
    
    for param_name, mask in masks.items():
        total_params += mask.numel()
        pruned_params += (mask == 0).sum().item()
    
    actual_ratio = pruned_params / total_params
    
    print(f"{config['name']}:")
    print(f"  Total parameters: {total_params:,}")
    print(f"  Pruned parameters: {pruned_params:,}")
    print(f"  Actual pruning ratio: {actual_ratio:.1%}")
    print()

## Step 7: Apply Pruning and Compare Models

In [ ]:
# Get 50% weight-level pruning mask
pruning_masks = fisher_hook.get_pruning_mask(pruning_ratio=0.5, granularity="weight")

# Apply masks to model
def apply_pruning_masks(model, masks):
    """Apply pruning masks to model parameters"""
    for name, param in model.named_parameters():
        if name in masks:
            param.data *= masks[name]

apply_pruning_masks(model, pruning_masks)
print("✓ Pruning masks applied to model")

# Test pruned model
model.eval()
test_input = torch.randn(4, 3, 32, 32)

with torch.no_grad():
    output = model(test_input)

print(f"\nPruned model inference:")
print(f"  Input shape: {test_input.shape}")
print(f"  Output shape: {output.shape}")
print(f"  ✓ Pruned model still works!")

## Step 8: Visualize Pruning Effect

In [ ]:
# Visualize pruning for first conv layer
conv1_weight = model.conv1.weight.data
conv1_mask = pruning_masks.get('conv1.weight', torch.ones_like(conv1_weight))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Original weights (first filter, first channel)
axes[0].imshow(conv1_weight[0, 0].cpu().numpy(), cmap='RdBu_r')
axes[0].set_title('Conv1 Filter (After Pruning)')
axes[0].axis('off')

# Mask
axes[1].imshow(conv1_mask[0, 0].cpu().numpy(), cmap='binary')
axes[1].set_title('Pruning Mask (White=Keep, Black=Pruned)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Count sparsity in conv1
sparsity = (conv1_mask == 0).float().mean().item()
print(f"Conv1 sparsity: {sparsity:.1%} of weights pruned")

## Step 9: Fine-Tuning After Pruning

After pruning, we should fine-tune to recover accuracy.

In [ ]:
# Fine-tune pruned model
print("Fine-tuning pruned model...")
model.train()
optimizer_ft = optim.SGD(model.parameters(), lr=0.001)  # Lower LR for fine-tuning

for batch_idx, (data, target) in enumerate(train_data[:10]):
    optimizer_ft.zero_grad()
    output = model(data)
    loss = criterion(output, target)
    loss.backward()
    
    # Re-apply masks after gradient step to keep pruned weights at zero
    optimizer_ft.step()
    apply_pruning_masks(model, pruning_masks)
    
    if (batch_idx + 1) % 5 == 0:
        print(f"  Batch {batch_idx + 1}/10, Loss: {loss.item():.4f}")

print("\n✓ Fine-tuning complete")

## Pruning Strategies Comparison

### Weight-Level Pruning (Unstructured)
- **Pros**: Maximum compression, fine-grained control
- **Cons**: Requires sparse matrix libraries for speedup
- **Best for**: Research, maximum compression

### Neuron-Level Pruning (Structured)
- **Pros**: Hardware-friendly, actual speedup on standard GPUs
- **Cons**: Less compression than weight-level
- **Best for**: Production deployment

### Channel-Level Pruning (Structured)
- **Pros**: Reduces memory and computation, easy to implement
- **Cons**: Coarse-grained, may hurt accuracy more
- **Best for**: Mobile deployment, edge devices

## Best Practices

1. **Gradual Pruning**: Start with 10-20%, gradually increase
2. **Iterative Pruning**: Prune → Fine-tune → Prune again
3. **Layer-wise Sensitivity**: Some layers are more sensitive, prune less
4. **Fine-tuning**: Always fine-tune after pruning
5. **Evaluation**: Monitor accuracy drop vs. compression ratio

## Recommended Workflow

```python
# 1. Train model normally
train_model(model, epochs=10)

# 2. Collect Fisher information
with HookManager() as manager:
    fisher_hook = FisherHook()
    manager.register(fisher_hook).apply_to_model(model)
    # Run a few batches
    
# 3. Generate pruning masks
masks = fisher_hook.get_pruning_mask(pruning_ratio=0.3)

# 4. Apply masks
apply_pruning_masks(model, masks)

# 5. Fine-tune
fine_tune(model, epochs=2, keep_masks=True)

# 6. Evaluate
evaluate(model)
```

## Next Steps

Try these exercises:

1. Apply pruning to a pre-trained model from torchvision
2. Compare different pruning ratios (30%, 50%, 70%)
3. Implement iterative pruning (prune 10% at a time)
4. Measure actual inference speedup with structured pruning
5. Combine with quantization for maximum compression

**Related**: Check `examples/use_case_llm_optimization.py` for a complete optimization pipeline